# IMAGINE Pipeline
## Denoising Autoencoder → Stable Diffusion Image Generation

**Run cells in order (1 → 7).**

> Cell 6 downloads Stable Diffusion (~4 GB, one-time only).

---
## Cell 1 — Install Dependencies *(skip if already installed)*

In [ ]:
import subprocess, sys
packages = ["torch", "torchvision", "diffusers", "transformers", "accelerate", "pillow", "matplotlib", "numpy"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"{pkg} already installed")
    except ImportError:
        print(f"📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


---
## Cell 2 — Imports & Device Setup

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import numpy as np
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"   GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:


---
## Cell 3 — Load Model (Exact Architecture from Training Notebook)

### **STUDENT EXERCISE 1: Define the Architecture**
Complete the `ResidualBlock` and the decoder of `RobustUNet`.
- For `Conv2d`, think about what kernel size and padding keeps the image size the same.
- For `ConvTranspose2d`, you need to upsample the features. Provide the correct input/output channels.


In [ ]:
# ── EXACT architecture copied from training notebook ─────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=___, padding=___)  # Fill in kernel & padding
        self.bn1   = nn.InstanceNorm2d(out_channels)  # InstanceNorm better for generative tasks
        self.relu  = nn.LeakyReLU(0.2, inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2   = nn.InstanceNorm2d(out_channels)
        # Skip connection: 1×1 conv when channels differ, Identity otherwise
        self.match = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        skip = self.match(x)
        out  = self.relu(self.bn1(self.conv1(x)))
        out  = self.bn2(self.conv2(out))
        return self.relu(out + skip)   # ReLU AFTER residual addition


class RobustUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1       = ResidualBlock(1,   64)
        self.enc2       = ResidualBlock(64,  128)
        self.enc3       = ResidualBlock(128, 256)
        self.pool       = nn.MaxPool2d(2)
        self.dropout    = nn.Dropout2d(0.2)
        # Bottleneck
        self.bottleneck = ResidualBlock(256, 512)
        # Decoder
        self.up3        = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3       = ResidualBlock(512, 256)  # 256 up + 256 skip
        self.up2        = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2       = ResidualBlock(256, 128)
        self.up1        = nn.ConvTranspose2d(128, 64,  2, stride=2)
        self.dec1       = ResidualBlock(128, 64)
        self.final      = nn.Conv2d(64, 1, 1)
        self.sigmoid    = nn.Sigmoid()

    def forward(self, x):
        # Encode
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        # Bridge (dropout before bottleneck)
        b  = self.bottleneck(self.pool(self.dropout(e3)))
        # Decode
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(self.dropout(d3))
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.sigmoid(self.final(d1))


MODEL_PATH = "robust_autoencoder_best.pth"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"'{MODEL_PATH}' not found. Place it in the same folder as this notebook.")

ae = RobustUNet().to(DEVICE)
ae.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
ae.eval()  # inference only — no gradients

print(f"Loaded: {MODEL_PATH}")
print(f"   Parameters : {sum(p.numel() for p in ae.parameters()):,}")
print(f"   Device     : {next(ae.parameters()).device}")


---
## Cell 4 — Set Your Noisy Image Path

**Edit the path below** to point to your noisy image, then run.

In [ ]:
# EDIT THIS: path to your already-noisy image
IMAGE_PATH = r"______________" #put your test image path here

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Not found: {IMAGE_PATH}")

print(f"Image found: {IMAGE_PATH}")


---
## Cell 5 — Denoise with Autoencoder

Loads your noisy image as-is (grayscale), passes it directly to the autoencoder — **no extra noise added**.

### **STUDENT EXERCISE 2: Model Inference**
Pass the `noisy_t` tensor through your autoencoder model (`ae`).


In [ ]:
# Load image as grayscale (no preprocessing, no extra noise)
noisy_pil = Image.open(IMAGE_PATH).convert("L")  # grayscale
orig_w, orig_h = noisy_pil.size
print(f"Image loaded: {orig_w}×{orig_h}px")

# Resize to 128×128 (model input size) and convert to tensor [0, 1]
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])
noisy_t = transform(noisy_pil).unsqueeze(0).to(DEVICE)  # [1, 1, 128, 128]

# Run autoencoder — inference only
print("Running autoencoder denoising...")
with torch.no_grad():
    denoised_t = ae(____)  # Pass the noisy tensor through the autoencoder model  # [1, 1, 128, 128]

# Convert back to numpy for display
noisy_np    = (noisy_t.squeeze().cpu().numpy() * 255).astype(np.uint8)
denoised_np = (denoised_t.squeeze().cpu().numpy() * 255).astype(np.uint8)

# Upscale denoised back to original resolution
denoised_pil  = Image.fromarray(denoised_np)
denoised_orig = denoised_pil.resize((orig_w, orig_h), Image.LANCZOS)
denoised_rgb  = denoised_orig.convert("RGB")  # Stable Diffusion needs RGB

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#1a1a2e')
axes[0].imshow(noisy_np,    cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy Input",    fontsize=14, color='white', fontweight='bold', pad=10)
axes[0].set_facecolor('#16213e')
axes[1].imshow(denoised_np, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Denoised Output", fontsize=14, color='white', fontweight='bold', pad=10)
axes[1].set_facecolor('#16213e')
for ax in axes: ax.axis("off")
plt.suptitle("Autoencoder Denoising Results", fontsize=15,
             fontweight="bold", color='white', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDenoising complete! Output: {orig_w}×{orig_h}")


---
## Cell 6 — Load Stable Diffusion v1.5

> First run downloads ~4 GB. After that it loads from local cache in seconds.

In [ ]:
print("Loading Stable Diffusion v1.5...")
print("   (First run: downloads ~4 GB — please be patient)")
print()

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to(DEVICE)

if DEVICE == "cuda":
    pipe.enable_attention_slicing()

print("Stable Diffusion v1.5 (img2img) ready!")
print(f"   Precision: {'float16 (GPU)' if DEVICE == 'cuda' else 'float32 (CPU)'}")


---
## Cell 7 — Set Prompt & Generate

Edit the settings below, then run.

- **`STRENGTH`** (0.1–0.95): lower = more faithful to the denoised sketch, higher = more creative
- **`GUIDANCE`**: how strictly the prompt is followed (7.5 is a good default)

### **STUDENT EXERCISE 3: Diffusion Magic**
Call the Stable Diffusion pipeline to generate the image. You need to provide the prompt, negative prompt, image, strength, and guidance scale.


In [ ]:
# EDIT THESE:
PROMPT          = " professional studio photograph of a vibrant ripe red apple, shiny skin, macro photography, high resolution, 8k, photorealistic, cinematic lighting"
NEGATIVE_PROMPT = "blurry, low quality, noisy, ugly, distorted, watermark"
STRENGTH        = 0.65   # 0.1 (keep sketch) → 0.95 (fully creative)
GUIDANCE        = 8.0
STEPS           = 30

print(f"Generating...")
print(f"   Prompt   : {PROMPT}")
print(f"   Strength : {STRENGTH}  |  Guidance : {GUIDANCE}  |  Steps : {STEPS}")
if DEVICE == "cpu":
    print("    CPU mode — this may take several minutes.")
print()

# SD v1.5 is optimised for 512×512
sd_input = denoised_rgb.resize((512, 512), Image.LANCZOS)

result_img = pipe(
    # Fill in the arguments for the pipeline
    prompt=____,
    negative_prompt=____,
    image=____,
    strength=____,
    guidance_scale=____,
    num_inference_steps=STEPS,
).images[0]

# Show all 3 stages
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0d0d1a')
panels = [
    (noisy_np,    "gray", "Noisy Input"),
    (denoised_np, "gray", "Denoised Sketch"),
    (result_img,  None,   "Generated Output"),
]
for ax, (img, cmap, title) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=13, color='white', fontweight='bold', pad=12)
    ax.axis("off")
    ax.set_facecolor('#1a1a2e')
plt.suptitle(f'Prompt: "{PROMPT}"', fontsize=11, color='#aaddff', style='italic', y=1.01)
plt.tight_layout()
plt.show()

out_path = "diffusion_output.png"
result_img.save(out_path)
